# Tool calling against a SQL database

Same mechanism as `tool_calling_basics.ipynb` (schema in → `tool_use` out → we execute → `tool_result` back in), just with real tools that hit a real database instead of `add`/`multiply`.

**Tool design choice: narrow, purpose-built tools** — each tool wraps one fixed, parameterized query (`get_customer_by_name(name)`, not a generic `run_sql_query(query)` that lets the model write arbitrary SQL). Two reasons:
- **Safety**: the model can only ever trigger the specific queries we've written, with parameters passed through placeholders (`?`) rather than string-substituted into SQL — the standard defense against SQL injection. A generic SQL tool would let the model (or anything that manages to influence its output) run arbitrary reads/writes.
- **It's the more common production pattern** — most real agents expose specific capabilities, not raw database access.

A generic `run_sql_query` tool is a reasonable stretch goal afterward, specifically *because* it's less safe — good setup for the prompt-injection/AgentDojo project next.

In [2]:
import json
import sqlite3

import anthropic
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL = "claude-opus-5"

## Step 1: build a small e-commerce database

Four tables so answering a real question requires joins/multiple lookups, not one flat filter: `customers`, `products`, `orders` (which customer, which date), `order_items` (which products, in which order, what quantity). In-memory SQLite — recreated fresh every time this cell runs, nothing persisted to disk.

In [3]:
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row  # lets us read rows like dicts

conn.executescript(
    """
    CREATE TABLE customers (
        id INTEGER PRIMARY KEY,
        name TEXT NOT NULL,
        email TEXT NOT NULL
    );

    CREATE TABLE products (
        id INTEGER PRIMARY KEY,
        name TEXT NOT NULL,
        price REAL NOT NULL
    );

    CREATE TABLE orders (
        id INTEGER PRIMARY KEY,
        customer_id INTEGER NOT NULL REFERENCES customers(id),
        order_date TEXT NOT NULL
    );

    CREATE TABLE order_items (
        id INTEGER PRIMARY KEY,
        order_id INTEGER NOT NULL REFERENCES orders(id),
        product_id INTEGER NOT NULL REFERENCES products(id),
        quantity INTEGER NOT NULL
    );

    INSERT INTO customers (id, name, email) VALUES
        (1, 'Alice Johnson', 'alice@example.com'),
        (2, 'Bob Smith', 'bob@example.com'),
        (3, 'Carol Lee', 'carol@example.com');

    INSERT INTO products (id, name, price) VALUES
        (1, 'Widget', 9.99),
        (2, 'Gadget', 19.99),
        (3, 'Gizmo', 14.99),
        (4, 'Doohickey', 4.99);

    INSERT INTO orders (id, customer_id, order_date) VALUES
        (1, 1, '2026-08-01'),
        (2, 1, '2026-08-15'),
        (3, 2, '2026-08-10');

    INSERT INTO order_items (order_id, product_id, quantity) VALUES
        (1, 1, 2),
        (1, 2, 1),
        (2, 3, 3),
        (3, 4, 5);
    """
)
conn.commit()
print("database ready")

database ready


Quick sanity check — run a query directly (no LLM involved) to confirm the data looks right before we build tools around it.

In [4]:
for row in conn.execute("SELECT * FROM orders"):
    print(dict(row))

{'id': 1, 'customer_id': 1, 'order_date': '2026-08-01'}
{'id': 2, 'customer_id': 1, 'order_date': '2026-08-15'}
{'id': 3, 'customer_id': 2, 'order_date': '2026-08-10'}


## Step 2: the real tool functions

Three narrow tools, each one fixed query. All use `?` placeholders — the argument the model provides is passed as a bound parameter, never string-concatenated into the SQL text.

Each returns a list of dicts (one per row) so results are easy to serialize back to the model as JSON.

In [5]:
def get_customer_by_name(name):
    rows = conn.execute(
        "SELECT id, name, email FROM customers WHERE name LIKE ?",
        (f"%{name}%",),
    ).fetchall()
    return [dict(row) for row in rows]


def get_orders_by_customer_id(customer_id):
    rows = conn.execute(
        "SELECT id, order_date FROM orders WHERE customer_id = ?",
        (customer_id,),
    ).fetchall()
    return [dict(row) for row in rows]


def get_order_items(order_id):
    rows = conn.execute(
        """
        SELECT p.name AS product_name, oi.quantity, p.price
        FROM order_items oi
        JOIN products p ON p.id = oi.product_id
        WHERE oi.order_id = ?
        """,
        (order_id,),
    ).fetchall()
    return [dict(row) for row in rows]

## Step 3: tool schemas

Same shape as `add`/`multiply` before — `name`, `description`, `input_schema`. The model only ever sees these descriptions, never the SQL itself, so they need to be clear about what each tool does and what it returns.

In [6]:
tools = [
    {
        "name": "get_customer_by_name",
        "description": "Look up a customer by (partial) name. Returns a list of matching customers with id, name, and email.",
        "input_schema": {
            "type": "object",
            "properties": {
                "name": {"type": "string", "description": "Full or partial customer name."},
            },
            "required": ["name"],
        },
    },
    {
        "name": "get_orders_by_customer_id",
        "description": "Get all orders placed by a customer, given their customer id. Returns a list of orders with id and order_date.",
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id": {"type": "integer"},
            },
            "required": ["customer_id"],
        },
    },
    {
        "name": "get_order_items",
        "description": "Get the line items in a specific order, given its order id. Returns a list of items with product_name, quantity, and price (price is per unit).",
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {"type": "integer"},
            },
            "required": ["order_id"],
        },
    },
]

tool_functions = {
    "get_customer_by_name": get_customer_by_name,
    "get_orders_by_customer_id": get_orders_by_customer_id,
    "get_order_items": get_order_items,
}

## Step 4: the tool loop

Identical to `run_tool_loop` from `tool_calling_basics.ipynb` — nothing database-specific about the loop itself, only the tools passed into it change. One difference: results are lists of dicts now (not a plain number), so we `json.dumps` them for the `tool_result` content instead of `str()`.

In [7]:
def run_tool_loop(messages, tools, tool_functions):
    while True:
        response = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            tools=tools,
            messages=messages,
        )
        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason != "tool_use":
            return response

        tool_use_blocks = [b for b in response.content if b.type == "tool_use"]
        tool_results = []
        for block in tool_use_blocks:
            print(f"  -> calling {block.name}({block.input})")
            result = tool_functions[block.name](**block.input)
            tool_results.append(
                {
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps(result),
                }
            )

        messages.append({"role": "user", "content": tool_results})

## Step 5: ask a question that needs all three tools, in order

This one is genuinely *sequential*, not parallel like the `add`/`multiply` example: the model can't get the order items without an order id, and can't get an order id without a customer id first. Each tool call depends on the previous one's result.

In [8]:
messages = [
    {
        "role": "user",
        "content": "What did Alice Johnson order, and how much did she spend in total?",
    }
]

final_response = run_tool_loop(messages, tools, tool_functions)

print("stop_reason:", final_response.stop_reason)
final_text = next(b.text for b in final_response.content if b.type == "text")
print("final answer:", final_text)

  -> calling get_customer_by_name({'name': 'Alice Johnson'})
  -> calling get_orders_by_customer_id({'customer_id': 1})
  -> calling get_order_items({'order_id': 1})
  -> calling get_order_items({'order_id': 2})
stop_reason: end_turn
final answer: Alice Johnson (alice@example.com) placed two orders:

**Order #1 — 2026-08-01**
| Product | Qty | Unit price | Subtotal |
|---|---|---|---|
| Widget | 2 | $9.99 | $19.98 |
| Gadget | 1 | $19.99 | $19.99 |
| | | **Order total** | **$39.97** |

**Order #2 — 2026-08-15**
| Product | Qty | Unit price | Subtotal |
|---|---|---|---|
| Gizmo | 3 | $14.99 | $44.97 |
| | | **Order total** | **$44.97** |

**Total spent: $84.94** across 2 orders (6 items total).


## Recap

Same loop as the `add` example, real tools: schema in → `tool_use` out → we run real parameterized SQL → `tool_result` back in → repeat until done. The model chained three separate tool calls (customer → orders → items) on its own, in the right order, purely from the tool descriptions and the results it got back — nothing in our code told it that order.

**Stretch goal:** add a fourth, generic `run_sql_query(query)` tool that lets the model write its own SQL against a schema description in the system prompt, and compare — what does it get you, and what new risk does it open up (a natural bridge into the prompt-injection/AgentDojo project).

## Stretch goal: the generic `run_sql_query` tool

Same question as Step 5, but one generic tool instead of three narrow ones — the model writes its own SQL, using a schema description we give it in the system prompt (`system=`, a new parameter we haven't used until now). Checking: does it write a sensible query, and does the total match the $84.94 we got before?

In [ ]:
sql_tools = [
    {
        "name": "run_sql_query",
        "description": "Run a read-only SQL SELECT query against the e-commerce database and return the results as a list of rows.",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "A SQL SELECT query."},
            },
            "required": ["query"],
        },
    }
]


def run_sql_query(query):
    rows = conn.execute(query).fetchall()
    return [dict(row) for row in rows]


sql_tool_functions = {"run_sql_query": run_sql_query}

schema_description = """You have access to a SQLite database with this schema:

customers(id, name, email)
products(id, name, price)
orders(id, customer_id, order_date)
order_items(id, order_id, product_id, quantity)

Use the run_sql_query tool to answer questions by writing SELECT queries against these tables."""

messages = [
    {
        "role": "user",
        "content": "What did Alice Johnson order, and how much did she spend in total?",
    }
]

while True:
    response = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        system=schema_description,
        tools=sql_tools,
        messages=messages,
    )
    messages.append({"role": "assistant", "content": response.content})

    if response.stop_reason != "tool_use":
        break

    tool_use_blocks = [b for b in response.content if b.type == "tool_use"]
    tool_results = []
    for block in tool_use_blocks:
        print(f"  -> SQL: {block.input['query']}")
        result = sql_tool_functions[block.name](**block.input)
        tool_results.append(
            {
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": json.dumps(result),
            }
        )

    messages.append({"role": "user", "content": tool_results})

print("stop_reason:", response.stop_reason)
final_text = next(b.text for b in response.content if b.type == "text")
print("final answer:", final_text)

  -> SQL: SELECT * FROM customers WHERE name LIKE '%Alice Johnson%'
  -> SQL: SELECT c.name, o.id AS order_id, o.order_date, p.name AS product, p.price, oi.quantity, (p.price * oi.quantity) AS line_total
FROM customers c
JOIN orders o ON o.customer_id = c.id
JOIN order_items oi ON oi.order_id = o.id
JOIN products p ON p.id = oi.product_id
WHERE c.name LIKE '%Alice Johnson%'
ORDER BY o.order_date, o.id
stop_reason: end_turn
final answer: **Alice Johnson** (alice@example.com) has placed 2 orders:

**Order #1 — 2026-08-01**
| Product | Price | Qty | Line Total |
|---|---|---|---|
| Widget | $9.99 | 2 | $19.98 |
| Gadget | $19.99 | 1 | $19.99 |
| | | | **$39.97** |

**Order #2 — 2026-08-15**
| Product | Price | Qty | Line Total |
|---|---|---|---|
| Gizmo | $14.99 | 3 | $44.97 |
| | | | **$44.97** |

**Total spent: $84.94** across 6 items (3 distinct products).
